# Data Lakehouse Tutorial - Part 3: Silver Layer Transformations

## Overview
This notebook demonstrates how to transform raw Bronze layer data into clean, validated Silver layer data.

### What is the Silver Layer?
- **Clean Data**: Validated, standardized, and deduplicated
- **Current State**: Latest version of each record (resolved CDC)
- **Quality Assured**: Data quality flags and validation rules
- **Business Ready**: Proper data types and business logic applied
- **Delta Format**: ACID transactions and schema evolution

### Key Transformations
1. **Deduplication**: Keep only the latest version of each record
2. **Data Cleaning**: Standardize formats, handle nulls
3. **Validation**: Apply business rules and quality checks
4. **Type Conversion**: Ensure proper data types
5. **Computed Columns**: Add derived fields

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
import pandas as pd
from datetime import datetime, timedelta

# Initialize Spark session with Delta Lake
spark = SparkSession.builder \
    .appName("SilverLayerTransformations") \
    .config("spark.master", "spark://spark-master:7077") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark session created: {spark.sparkContext.appName}")
print(f"Delta Lake enabled: {'io.delta' in str(spark.conf.get('spark.sql.extensions'))}")

## 1. Read Bronze Layer Data

First, let's load the raw data from our Bronze layer:

In [ ]:
def load_bronze_data(table_name):
    """
    Load data from Bronze layer for transformation.
    """
    try:
        bronze_path = f"s3a://bronze/{table_name}/"
        df = spark.read.format("parquet").load(bronze_path)
        
        record_count = df.count()
        print(f"📥 Loaded {record_count:,} records from Bronze {table_name}")
        
        if record_count > 0:
            # Show basic info
            print(f"   Columns: {len(df.columns)}")
            print(f"   Date Range: {df.agg(min('ingestion_timestamp'), max('ingestion_timestamp')).collect()[0]}")
            
            # Show operation distribution
            op_dist = df.groupBy("debezium_op").count().collect()
            print(f"   Operations: {dict((row['debezium_op'], row['count']) for row in op_dist)}")
        
        return df
        
    except Exception as e:
        print(f"❌ Error loading Bronze data for {table_name}: {e}")
        return None

# Load Bronze data for orders
bronze_orders = load_bronze_data("orders")
bronze_customers = load_bronze_data("customers")
bronze_products = load_bronze_data("products")

## 2. Understand CDC Resolution

The most important transformation in Silver layer is resolving CDC operations to get the current state:

In [ ]:
def demonstrate_cdc_resolution(df, table_name, id_column):
    """
    Demonstrate how CDC operations are resolved to get current state.
    """
    if df is None or df.count() == 0:
        print(f"No data available for CDC resolution demo in {table_name}")
        return None
    
    print(f"\n=== CDC RESOLUTION DEMO: {table_name.upper()} ===")
    
    # Find a record with multiple CDC events to demonstrate
    records_with_changes = df.groupBy(id_column).count().filter(col("count") > 1)
    
    if records_with_changes.count() > 0:
        sample_id = records_with_changes.first()[id_column]
        print(f"\n📋 Example: All CDC events for {id_column} = {sample_id}")
        
        # Show all events for this record
        sample_events = df.filter(col(id_column) == sample_id).orderBy("debezium_ts_ms")
        
        display_columns = [id_column, "debezium_op", "debezium_ts_ms"]
        # Add some business columns based on table type
        if table_name == "orders":
            display_columns.extend(["status", "total_cents"])
        elif table_name == "customers":
            display_columns.extend(["email", "full_name"])
        elif table_name == "products":
            display_columns.extend(["name", "price_cents", "active"])
        
        available_columns = [c for c in display_columns if c in df.columns]
        sample_events.select(available_columns).show(truncate=False)
        
        print("\n🔄 Resolution Process:")
        print("   1. Order by debezium_ts_ms (timestamp)")
        print("   2. Keep only the LATEST record for each ID")
        print("   3. Filter out DELETE operations (debezium_op = 'd')")
        print("   4. Result = Current state of the record")
    
    # Demonstrate the resolution process
    print(f"\n📊 Before Resolution:")
    print(f"   Total CDC Events: {df.count():,}")
    
    # Apply deduplication logic
    window_spec = Window.partitionBy(id_column).orderBy(desc("debezium_ts_ms"))
    
    # Get latest record for each ID
    latest_records = df.withColumn("row_num", row_number().over(window_spec)) \
                       .filter(col("row_num") == 1) \
                       .drop("row_num")
    
    # Filter out deletes
    current_state = latest_records.filter(col("debezium_op") != "d")
    
    print(f"\n📊 After Resolution:")
    print(f"   Latest Records (including deletes): {latest_records.count():,}")
    print(f"   Current Active Records: {current_state.count():,}")
    
    # Show operation distribution in final state
    final_ops = current_state.groupBy("debezium_op").count().collect()
    print(f"   Final Operations: {dict((row['debezium_op'], row['count']) for row in final_ops)}")
    
    return current_state

# Demonstrate CDC resolution for orders
if bronze_orders:
    resolved_orders = demonstrate_cdc_resolution(bronze_orders, "orders", "order_id")

## 3. Data Cleaning and Validation

Now let's apply comprehensive data cleaning and validation rules:

In [ ]:
def clean_orders_data(df):
    """
    Apply comprehensive cleaning and validation to orders data.
    """
    if df is None or df.count() == 0:
        print("No orders data to clean")
        return None
    
    print("\n=== ORDERS DATA CLEANING ===")
    
    # Step 1: Deduplication (get latest state)
    window_spec = Window.partitionBy("order_id").orderBy(desc("debezium_ts_ms"))
    df_deduped = df.withColumn("row_num", row_number().over(window_spec)) \
                   .filter(col("row_num") == 1) \
                   .drop("row_num")
    
    # Step 2: Filter out soft deletes
    df_active = df_deduped.filter(col("debezium_op") != "d")
    
    print(f"   ✅ Deduplication: {df.count():,} → {df_active.count():,} records")
    
    # Step 3: Data type conversion and cleaning
    df_clean = df_active.select(
        # Primary key and foreign keys
        col("order_id").cast(LongType()).alias("order_id"),
        col("customer_id").cast(LongType()).alias("customer_id"),
        col("ship_to_address_id").cast(LongType()).alias("ship_to_address_id"),
        
        # Clean and standardize status
        when(col("status").isNull(), "UNKNOWN")
        .when(upper(trim(col("status"))) == "PLACED", "PLACED")
        .when(upper(trim(col("status"))) == "PAID", "PAID")
        .when(upper(trim(col("status"))) == "FULFILLED", "FULFILLED")
        .when(upper(trim(col("status"))) == "CANCELED", "CANCELED")
        .otherwise("UNKNOWN").alias("status"),
        
        # Clean currency
        when(col("currency").isNull(), "USD")
        .otherwise(upper(trim(col("currency")))).alias("currency"),
        
        # Validate and clean monetary amounts (ensure non-negative)
        when(col("subtotal_cents").isNull() | (col("subtotal_cents") < 0), 0)
        .otherwise(col("subtotal_cents").cast(IntegerType())).alias("subtotal_cents"),
        
        when(col("shipping_cents").isNull() | (col("shipping_cents") < 0), 0)
        .otherwise(col("shipping_cents").cast(IntegerType())).alias("shipping_cents"),
        
        when(col("tax_cents").isNull() | (col("tax_cents") < 0), 0)
        .otherwise(col("tax_cents").cast(IntegerType())).alias("tax_cents"),
        
        when(col("total_cents").isNull() | (col("total_cents") < 0), 0)
        .otherwise(col("total_cents").cast(IntegerType())).alias("total_cents"),
        
        # Convert timestamps
        to_timestamp(col("created_at")).alias("created_at"),
        to_timestamp(col("ingestion_timestamp")).alias("ingestion_timestamp"),
        
        # Add computed columns
        ((col("subtotal_cents") + col("shipping_cents") + col("tax_cents")) / 100.0).alias("calculated_total_dollars"),
        (col("total_cents") / 100.0).alias("total_dollars"),
        
        # Add data quality flags
        when(
            (col("total_cents") != (col("subtotal_cents") + col("shipping_cents") + col("tax_cents"))) |
            col("customer_id").isNull() |
            col("ship_to_address_id").isNull() |
            (col("total_cents") < 0),
            "FAILED"
        ).otherwise("PASSED").alias("data_quality_status"),
        
        # Metadata columns
        col("debezium_op").alias("cdc_operation"),
        col("debezium_ts_ms").alias("cdc_timestamp_ms"),
        current_timestamp().alias("silver_processed_at")
    )
    
    # Show cleaning results
    print(f"   ✅ Data cleaning completed")
    
    # Quality check summary
    quality_dist = df_clean.groupBy("data_quality_status").count().collect()
    quality_summary = dict((row['data_quality_status'], row['count']) for row in quality_dist)
    print(f"   📊 Quality Status: {quality_summary}")
    
    # Show some statistics
    if "total_dollars" in df_clean.columns:
        stats = df_clean.agg(
            avg("total_dollars").alias("avg_total"),
            min("total_dollars").alias("min_total"),
            max("total_dollars").alias("max_total")
        ).collect()[0]
        
        print(f"   💰 Order Values: Avg=${stats['avg_total']:.2f}, Min=${stats['min_total']:.2f}, Max=${stats['max_total']:.2f}")
    
    return df_clean

# Clean orders data
if bronze_orders:
    clean_orders = clean_orders_data(bronze_orders)
    
    if clean_orders:
        print("\n📋 Sample cleaned data:")
        clean_orders.select(
            "order_id", "status", "total_dollars", "data_quality_status", "silver_processed_at"
        ).show(5, truncate=False)

In [ ]:
def clean_customers_data(df):
    """
    Clean customers data with validation rules.
    """
    if df is None or df.count() == 0:
        print("No customers data to clean")
        return None
    
    print("\n=== CUSTOMERS DATA CLEANING ===")
    
    # Deduplication and active records
    window_spec = Window.partitionBy("customer_id").orderBy(desc("debezium_ts_ms"))
    df_active = df.withColumn("row_num", row_number().over(window_spec)) \
                  .filter(col("row_num") == 1) \
                  .drop("row_num") \
                  .filter(col("debezium_op") != "d")
    
    df_clean = df_active.select(
        col("customer_id").cast(LongType()).alias("customer_id"),
        
        # Clean email (lowercase, trim)
        when(col("email").isNull(), "unknown@unknown.com")
        .otherwise(trim(lower(col("email")))).alias("email"),
        
        # Clean full_name (trim)
        when(col("full_name").isNull(), "Unknown Customer")
        .otherwise(trim(col("full_name"))).alias("full_name"),
        
        # Clean phone (remove non-digits except + and -)
        when(col("phone").isNull(), None)
        .otherwise(regexp_replace(col("phone"), "[^+\\-0-9]", "")).alias("phone"),
        
        to_timestamp(col("created_at")).alias("created_at"),
        to_timestamp(col("ingestion_timestamp")).alias("ingestion_timestamp"),
        
        # Email validation
        when(
            col("email").rlike("^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\.[a-zA-Z]{2,}$") &
            col("full_name").isNotNull() &
            (col("full_name") != ""),
            "PASSED"
        ).otherwise("FAILED").alias("data_quality_status"),
        
        col("debezium_op").alias("cdc_operation"),
        col("debezium_ts_ms").alias("cdc_timestamp_ms"),
        current_timestamp().alias("silver_processed_at")
    )
    
    print(f"   ✅ Cleaned {df_clean.count():,} customer records")
    
    return df_clean

def clean_products_data(df):
    """
    Clean products data with validation rules.
    """
    if df is None or df.count() == 0:
        print("No products data to clean")
        return None
    
    print("\n=== PRODUCTS DATA CLEANING ===")
    
    # Deduplication and active records
    window_spec = Window.partitionBy("product_id").orderBy(desc("debezium_ts_ms"))
    df_active = df.withColumn("row_num", row_number().over(window_spec)) \
                  .filter(col("row_num") == 1) \
                  .drop("row_num") \
                  .filter(col("debezium_op") != "d")
    
    df_clean = df_active.select(
        col("product_id").cast(LongType()).alias("product_id"),
        
        # Clean SKU (uppercase, trim)
        when(col("sku").isNull(), "UNKNOWN-SKU")
        .otherwise(trim(upper(col("sku")))).alias("sku"),
        
        # Clean product name
        when(col("name").isNull(), "Unknown Product")
        .otherwise(trim(col("name"))).alias("name"),
        
        # Validate price
        when(col("price_cents").isNull() | (col("price_cents") < 0), 0)
        .otherwise(col("price_cents").cast(IntegerType())).alias("price_cents"),
        
        (col("price_cents") / 100.0).alias("price_dollars"),
        
        # Clean active flag
        when(col("active").isNull(), False)
        .otherwise(col("active").cast(BooleanType())).alias("active"),
        
        # Data quality validation
        when(
            col("sku").isNotNull() &
            col("name").isNotNull() &
            (col("name") != "") &
            (col("price_cents") >= 0),
            "PASSED"
        ).otherwise("FAILED").alias("data_quality_status"),
        
        col("debezium_op").alias("cdc_operation"),
        col("debezium_ts_ms").alias("cdc_timestamp_ms"),
        current_timestamp().alias("silver_processed_at")
    )
    
    print(f"   ✅ Cleaned {df_clean.count():,} product records")
    
    return df_clean

# Clean other tables
clean_customers = clean_customers_data(bronze_customers) if bronze_customers else None
clean_products = clean_products_data(bronze_products) if bronze_products else None

## 4. Data Quality Analysis

Let's analyze the data quality of our cleaned Silver layer data:

In [ ]:
def analyze_data_quality(df, table_name):
    """
    Perform comprehensive data quality analysis.
    """
    if df is None or df.count() == 0:
        print(f"No data for quality analysis: {table_name}")
        return
    
    print(f"\n=== DATA QUALITY ANALYSIS: {table_name.upper()} ===")
    
    total_records = df.count()
    print(f"📊 Total Records: {total_records:,}")
    
    # Quality status distribution
    if "data_quality_status" in df.columns:
        quality_dist = df.groupBy("data_quality_status").count().orderBy("count", ascending=False)
        print("\n🏷️  Data Quality Distribution:")
        quality_dist.show()
        
        # Calculate pass rate
        passed_count = df.filter(col("data_quality_status") == "PASSED").count()
        pass_rate = (passed_count / total_records) * 100
        print(f"   ✅ Quality Pass Rate: {pass_rate:.1f}% ({passed_count:,} of {total_records:,} records)")
        
        if pass_rate < 95:
            print("   ⚠️  Quality pass rate is below 95% - investigate failed records")
            
            # Show examples of failed records
            failed_records = df.filter(col("data_quality_status") == "FAILED")
            if failed_records.count() > 0:
                print("\n❌ Sample Failed Records:")
                failed_records.limit(3).show(truncate=False)
    
    # Completeness analysis
    print("\n📋 Completeness Analysis:")
    for column in df.columns:
        if column not in ['silver_processed_at', 'cdc_operation', 'cdc_timestamp_ms']:
            null_count = df.filter(col(column).isNull()).count()
            completeness = ((total_records - null_count) / total_records) * 100
            
            status = "✅" if completeness == 100 else "⚠️" if completeness >= 90 else "❌"
            print(f"   {status} {column}: {completeness:.1f}% complete")
    
    # Table-specific validations
    if table_name == "orders":
        print("\n💰 Orders-specific Quality Checks:")
        
        # Check for negative amounts
        negative_totals = df.filter(col("total_dollars") < 0).count()
        print(f"   💸 Negative total amounts: {negative_totals}")
        
        # Check for unrealistic amounts
        very_high_amounts = df.filter(col("total_dollars") > 10000).count()
        print(f"   💎 Very high amounts (>$10k): {very_high_amounts}")
        
        # Status distribution
        status_dist = df.groupBy("status").count().orderBy("count", ascending=False)
        print("\n📈 Order Status Distribution:")
        status_dist.show()
        
    elif table_name == "customers":
        print("\n👥 Customers-specific Quality Checks:")
        
        if "email" in df.columns:
            # Check for email duplicates
            duplicate_emails = df.groupBy("email").count().filter(col("count") > 1).count()
            print(f"   📧 Duplicate email addresses: {duplicate_emails}")
            
            # Check for placeholder emails
            placeholder_emails = df.filter(col("email").rlike("unknown|test|example")).count()
            print(f"   🎭 Placeholder email addresses: {placeholder_emails}")
    
    elif table_name == "products":
        print("\n📦 Products-specific Quality Checks:")
        
        if "price_dollars" in df.columns:
            # Price analysis
            price_stats = df.agg(
                avg("price_dollars").alias("avg_price"),
                min("price_dollars").alias("min_price"),
                max("price_dollars").alias("max_price")
            ).collect()[0]
            
            print(f"   💰 Price range: ${price_stats['min_price']:.2f} - ${price_stats['max_price']:.2f} (avg: ${price_stats['avg_price']:.2f})")
            
            free_products = df.filter(col("price_dollars") == 0).count()
            print(f"   🆓 Free products (price = $0): {free_products}")
        
        if "active" in df.columns:
            active_dist = df.groupBy("active").count().collect()
            active_summary = dict((row['active'], row['count']) for row in active_dist)
            print(f"   📊 Active/Inactive: {active_summary}")

# Analyze quality for all cleaned tables
for df, name in [(clean_orders, "orders"), (clean_customers, "customers"), (clean_products, "products")]:
    if df:
        analyze_data_quality(df, name)

## 5. Write to Silver Layer (Delta Format)

Now let's save our cleaned data to the Silver layer using Delta format:

In [ ]:
def write_to_silver(df, table_name, mode="overwrite"):
    """
    Write cleaned data to Silver layer using Delta format.
    """
    if df is None or df.count() == 0:
        print(f"❌ No data to write for {table_name}")
        return False
    
    try:
        silver_path = f"s3a://silver/{table_name}"
        
        # Add partition columns for efficient queries
        current_date = datetime.now()
        df_partitioned = df.withColumn("year", lit(current_date.year)) \
                          .withColumn("month", lit(f"{current_date.month:02d}")) \
                          .withColumn("day", lit(f"{current_date.day:02d}"))
        
        print(f"\n💾 Writing {df_partitioned.count():,} records to Silver layer: {table_name}")
        
        # Write as Delta table with partitioning
        df_partitioned.write \
            .format("delta") \
            .mode(mode) \
            .option("mergeSchema", "true") \
            .partitionBy("year", "month", "day") \
            .save(silver_path)
        
        print(f"   ✅ Successfully written to {silver_path}")
        
        # Verify the write
        verification_df = spark.read.format("delta").load(silver_path)
        written_count = verification_df.count()
        print(f"   🔍 Verification: {written_count:,} records in Silver layer")
        
        return True
        
    except Exception as e:
        print(f"❌ Error writing to Silver layer for {table_name}: {e}")
        return False

# Write all cleaned data to Silver layer
tables_to_write = [
    (clean_orders, "orders"),
    (clean_customers, "customers"),
    (clean_products, "products")
]

successful_writes = []
for df, table_name in tables_to_write:
    if df is not None:
        success = write_to_silver(df, table_name)
        if success:
            successful_writes.append(table_name)

print(f"\n✅ Successfully wrote {len(successful_writes)} tables to Silver layer: {', '.join(successful_writes)}")

## 6. Delta Lake Features Demonstration

Let's explore some key Delta Lake features that make Silver layer powerful:

In [ ]:
def demonstrate_delta_features(table_name="orders"):
    """
    Demonstrate key Delta Lake features.
    """
    silver_path = f"s3a://silver/{table_name}"
    
    try:
        print(f"\n=== DELTA LAKE FEATURES: {table_name.upper()} ===")
        
        # Read Delta table
        df = spark.read.format("delta").load(silver_path)
        
        # 1. Schema information
        print("\n1️⃣ Schema Information:")
        print(f"   Columns: {len(df.columns)}")
        print(f"   Records: {df.count():,}")
        
        # Show schema
        print("\n📋 Schema:")
        df.printSchema()
        
        # 2. Transaction History
        print("\n2️⃣ Transaction History:")
        from delta.tables import DeltaTable
        
        delta_table = DeltaTable.forPath(spark, silver_path)
        history = delta_table.history()
        
        print("   Recent transactions:")
        history.select("version", "timestamp", "operation", "operationMetrics").show(5, truncate=False)
        
        # 3. Table Details
        print("\n3️⃣ Table Details:")
        details = delta_table.detail()
        detail_row = details.collect()[0]
        
        print(f"   Format: {detail_row['format']}")
        print(f"   Partition Columns: {detail_row['partitionColumns']}")
        print(f"   Number of Files: {detail_row['numFiles']}")
        print(f"   Size in Bytes: {detail_row['sizeInBytes']:,}")
        
        # 4. Partition Information
        print("\n4️⃣ Partition Information:")
        if "year" in df.columns and "month" in df.columns:
            partition_summary = df.groupBy("year", "month", "day").count().orderBy("year", "month", "day")
            print("   Records per partition:")
            partition_summary.show()
        
        # 5. Time Travel (if multiple versions exist)
        print("\n5️⃣ Time Travel Capability:")
        version_count = history.count()
        print(f"   Available versions: {version_count}")
        
        if version_count > 1:
            print("   You can query previous versions using:")
            print("   spark.read.format('delta').option('versionAsOf', 0).load(path)")
            print("   spark.read.format('delta').option('timestampAsOf', '2024-09-14').load(path)")
        
        # 6. Data Quality Summary
        print("\n6️⃣ Current Data Quality:")
        if "data_quality_status" in df.columns:
            quality_summary = df.groupBy("data_quality_status").count().collect()
            for row in quality_summary:
                status = row['data_quality_status']
                count = row['count']
                percentage = (count / df.count()) * 100
                emoji = "✅" if status == "PASSED" else "❌"
                print(f"   {emoji} {status}: {count:,} records ({percentage:.1f}%)")
        
        return True
        
    except Exception as e:
        print(f"❌ Error demonstrating Delta features for {table_name}: {e}")
        return False

# Demonstrate Delta features for each successful table
for table_name in successful_writes:
    demonstrate_delta_features(table_name)
    print("\n" + "="*60)

## 7. Silver Layer Query Patterns

Let's demonstrate common query patterns for Silver layer data:

In [ ]:
def demonstrate_silver_queries():
    """
    Demonstrate common Silver layer query patterns.
    """
    print("\n=== SILVER LAYER QUERY PATTERNS ===")
    
    try:
        # Load Silver tables
        orders = spark.read.format("delta").load("s3a://silver/orders")
        customers = spark.read.format("delta").load("s3a://silver/customers")
        products = spark.read.format("delta").load("s3a://silver/products")
        
        print(f"📊 Loaded tables - Orders: {orders.count():,}, Customers: {customers.count():,}, Products: {products.count():,}")
        
        # 1. High-quality data filtering
        print("\n1️⃣ High-Quality Data Query:")
        quality_orders = orders.filter(col("data_quality_status") == "PASSED")
        print(f"   High-quality orders: {quality_orders.count():,} of {orders.count():,}")
        
        # 2. Business analytics queries
        print("\n2️⃣ Business Analytics Queries:")
        
        # Order status analysis
        status_analysis = quality_orders.groupBy("status").agg(
            count("*").alias("order_count"),
            avg("total_dollars").alias("avg_value"),
            sum("total_dollars").alias("total_value")
        ).orderBy(desc("total_value"))
        
        print("   📈 Orders by Status:")
        status_analysis.show()
        
        # Daily order trends (if we have enough data)
        if "created_at" in orders.columns:
            daily_trends = quality_orders.withColumn("order_date", date_format("created_at", "yyyy-MM-dd")) \
                                        .groupBy("order_date").agg(
                                            count("*").alias("orders"),
                                            sum("total_dollars").alias("revenue")
                                        ).orderBy("order_date")
            
            print("   📅 Daily Order Trends:")
            daily_trends.show(10)
        
        # 3. Join operations (analytical joins)
        print("\n3️⃣ Analytical Joins:")
        
        # Customer order summary
        customer_summary = quality_orders.join(customers, "customer_id", "inner") \
                                        .groupBy("customer_id", "full_name", "email").agg(
                                            count("*").alias("total_orders"),
                                            sum("total_dollars").alias("total_spent"),
                                            avg("total_dollars").alias("avg_order_value")
                                        ).orderBy(desc("total_spent"))
        
        print("   👥 Top Customers by Spending:")
        customer_summary.show(10)
        
        # 4. Data quality monitoring queries
        print("\n4️⃣ Data Quality Monitoring:")
        
        # Quality trends over time
        quality_trends = orders.withColumn("process_date", date_format("silver_processed_at", "yyyy-MM-dd")) \
                              .groupBy("process_date", "data_quality_status").count() \
                              .orderBy("process_date", "data_quality_status")
        
        print("   📊 Quality Trends:")
        quality_trends.show()
        
        # Failed record analysis
        failed_orders = orders.filter(col("data_quality_status") == "FAILED")
        if failed_orders.count() > 0:
            print("\n   ❌ Failed Records Analysis:")
            # Show common failure patterns
            failed_orders.select("order_id", "status", "total_dollars", "calculated_total_dollars").show(5)
        
        # 5. Advanced analytics preparation
        print("\n5️⃣ Advanced Analytics Examples:")
        
        # Customer segmentation preparation
        customer_segments = customer_summary.withColumn(
            "customer_segment",
            when(col("total_spent") > 1000, "High Value")
            .when(col("total_spent") > 500, "Medium Value")
            .when(col("total_spent") > 100, "Regular")
            .otherwise("Low Value")
        )
        
        segment_distribution = customer_segments.groupBy("customer_segment").agg(
            count("*").alias("customers"),
            avg("total_spent").alias("avg_spent")
        ).orderBy(desc("avg_spent"))
        
        print("   🎯 Customer Segmentation:")
        segment_distribution.show()
        
        return True
        
    except Exception as e:
        print(f"❌ Error in Silver query demonstration: {e}")
        return False

# Run query demonstrations if we have data
if successful_writes:
    demonstrate_silver_queries()

## Summary and Next Steps

### What We Accomplished in Silver Layer:

1. **CDC Resolution**: Converted change events to current state
2. **Data Cleaning**: Applied validation and standardization rules
3. **Quality Assurance**: Added data quality flags and validation
4. **Delta Format**: Leveraged ACID transactions and schema evolution
5. **Partitioning**: Optimized for analytical queries

### Key Silver Layer Benefits:
- **Reliability**: ACID transactions ensure data consistency
- **Performance**: Partitioned Delta tables for fast queries
- **Quality**: Built-in validation and quality monitoring
- **Evolution**: Schema changes handled gracefully
- **Time Travel**: Historical versions available for analysis

### Silver Layer Best Practices:
1. ✅ Always include data quality flags
2. ✅ Implement proper deduplication logic
3. ✅ Use consistent data types and formats
4. ✅ Add computed columns for common calculations
5. ✅ Partition by commonly filtered columns
6. ✅ Monitor and track data quality metrics

### Next Steps:
1. **Part 4**: Build Gold layer aggregations and business views
2. **Part 5**: Advanced analytics and machine learning preparation
3. **Part 6**: Monitoring, alerting, and operational excellence

In [ ]:
# Cleanup
spark.stop()
print("\n✅ Silver layer tutorial complete. Spark session stopped.")